# Recognition + Retrieval Playground — Google Colab

A sandbox for the **first two stages only** — recognition (what the LLM sees) and retrieval
(what Open Food Facts / Open Products Facts have). Runs live in Colab (open network). Steps:
install deps → load the code → set your key → probe images or the catalogue directly.

> Runtime → *Run all* after uploading the code (Step 2) and setting your key (Step 3).

### Step 1 — install dependencies

In [ ]:
!pip install -q requests pillow opencv-python-headless imagehash pandas matplotlib python-dotenv

### Step 2 — load the code

Run the cell, then upload `snap-to-sell.zip` (provided alongside this notebook, or zip your local
`snap-to-sell/` folder). Or uncomment the git clone line if you've pushed the repo.

In [ ]:
import os, sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

# Already inside the repo (local Jupyter run from snap-to-sell/)? Just fix imports — no zip, no upload.
if os.path.isdir("src/snap_to_sell"):
    pass
elif os.path.isdir("snap-to-sell/src/snap_to_sell"):
    os.chdir("snap-to-sell")
elif _in_colab():
    import zipfile, io
    from google.colab import files
    up = files.upload()                      # choose snap-to-sell.zip
    fn = next(iter(up))
    zipfile.ZipFile(io.BytesIO(up[fn])).extractall(".")
    os.chdir("snap-to-sell")
    # --- OR clone from GitHub instead of uploading a zip ---
    # !git clone https://github.com/<your-org>/MIA5100_group_project.git
    # !cp -r MIA5100_group_project/snap-to-sell . && cd snap-to-sell
else:
    raise RuntimeError("Run this notebook from inside the snap-to-sell/ folder, "
                       "or provide snap-to-sell.zip in Colab.")

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print("test images:", [f for f in os.listdir("data/testset") if f.endswith(('.jpg', '.jpeg'))])

Upload step skipped: No module named 'google.colab'


FileNotFoundError: [Errno 2] No such file or directory: 'snap-to-sell'

### Step 3 — configuration

Set the provider, model, key and strict-mode. The key is entered via `getpass` (masked) if you
leave it blank. Set `SNAP_STRICT_PROVIDER = True` to force the real model (no sidecar) so you see
the true LLM output.

In [ ]:
import os, getpass

# Blank = use whatever is in your .env (recommended). Fill a field ONLY to override .env for this run.
SNAP_LLM_PROVIDER     = ""    # "openai" | "anthropic" | "gemini" | ""  -> blank uses .env / auto-detect
SNAP_OPENAI_MODEL     = ""    # e.g. "gpt-4o-mini"; blank -> .env / default
SNAP_RETRIEVE_BACKEND = ""    # "off" (Open Food Facts) | "shopping" (Serper); blank -> .env
OPENAI_API_KEY        = ""    # paste to override .env; blank -> use .env
SERPER_API_KEY        = ""    # blank -> use .env  (needed for backend="shopping")
SNAP_STRICT_PROVIDER  = None  # True / False to override; None -> use .env

# --- only non-empty cell values override .env ---
for _k, _v in {
    "SNAP_LLM_PROVIDER": SNAP_LLM_PROVIDER,
    "SNAP_OPENAI_MODEL": SNAP_OPENAI_MODEL,
    "SNAP_RETRIEVE_BACKEND": SNAP_RETRIEVE_BACKEND,
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "SERPER_API_KEY": SERPER_API_KEY,
}.items():
    if _v:
        os.environ[_k] = _v
if SNAP_STRICT_PROVIDER is not None:
    os.environ["SNAP_STRICT_PROVIDER"] = "1" if SNAP_STRICT_PROVIDER else ""

# config auto-loads the nearest .env (override=False), so keys in .env are picked up here.
from src.snap_to_sell import config
config.refresh()

# Prompt for a key ONLY if the chosen provider is OpenAI and no key came from the cell or .env.
_provider = (os.environ.get("SNAP_LLM_PROVIDER") or config.LLM_PROVIDER or "").lower()
if _provider == "openai" and not config.OPENAI_API_KEY:
    try:
        _k = getpass.getpass("OPENAI_API_KEY (blank = offline): ")
    except Exception:
        _k = ""
    if _k:
        os.environ["OPENAI_API_KEY"] = _k
        config.refresh()

# Convenience: with the shopping backend, show the clean retailer image (unless you set SNAP_IMAGE_MODE).
if config.RETRIEVE_BACKEND == "shopping" and not os.environ.get("SNAP_IMAGE_MODE"):
    os.environ["SNAP_IMAGE_MODE"] = "swap"
    config.refresh()

print("retrieval:", config.RETRIEVE_BACKEND)
print("provider :", config.active_provider() or "offline (no key)")
print("model    :", config.OPENAI_MODEL)
print("strict   :", config.STRICT_PROVIDER)

### Step 4 — helpers

In [ ]:
import os, base64, mimetypes, html as _html
from dataclasses import asdict
import requests
from src.snap_to_sell import config, trace
from src.snap_to_sell.capture import preprocess
from src.snap_to_sell.recognise import recognise
from src.snap_to_sell.retrieve import retrieve
from IPython.display import HTML, display

def _img_src(ref):
    if not ref:
        return ""
    if str(ref).startswith("http"):
        return ref
    try:
        mime = mimetypes.guess_type(ref)[0] or "image/jpeg"
        with open(ref, "rb") as f:
            return f"data:{mime};base64," + base64.b64encode(f.read()).decode()
    except Exception:
        return ""

_CSS = """<style>
.pg-card{display:flex;gap:14px;border:1px solid #ececf0;border-radius:14px;padding:12px;margin:10px 0;
  box-shadow:0 2px 10px rgba(20,20,40,.06);font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;}
.pg-photo{flex:0 0 150px;}
.pg-photo img{width:150px;height:150px;object-fit:contain;background:#f6f6f8;border-radius:10px;}
.pg-photo .cap{font-size:11px;color:#8a8a99;text-align:center;margin-top:4px;}
.pg-col{flex:1;min-width:180px;}
.pg-h{font-size:11px;letter-spacing:.06em;text-transform:uppercase;color:#8a8a99;font-weight:700;margin-bottom:4px;}
.pg-row{font-size:12.5px;color:#26263a;line-height:1.5;}
.pg-row b{color:#555;font-weight:600;}
.pg-badge{display:inline-block;font-size:10.5px;font-weight:700;padding:2px 9px;border-radius:20px;margin-bottom:6px;}
.pg-hit{background:#e6f7ee;color:#0a7d3c;}.pg-miss{background:#fff4e0;color:#b9770a;}.pg-demo{background:#eef0ff;color:#4a4ad0;}
.pg-thumb{width:90px;height:90px;object-fit:contain;background:#f6f6f8;border-radius:8px;margin-top:6px;}
.pg-log{margin:2px 0 12px;font-size:11px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;}
.pg-log summary{cursor:pointer;color:#4a4ad0;font-weight:700;}
.pg-log pre{max-height:260px;overflow:auto;background:#f7f7fb;border:1px solid #ececf0;border-radius:8px;padding:8px;font-size:10px;white-space:pre-wrap;}
</style>"""


def _log_html(path):
    lp = os.path.join(os.path.dirname(path), os.path.splitext(os.path.basename(path))[0] + "-log.txt")
    if not os.path.exists(lp):
        return ""
    try:
        txt = open(lp, encoding="utf-8").read()
    except Exception:
        return ""
    return ('<details class="pg-log"><summary>view processing log (' + _html.escape(os.path.basename(lp))
            + ')</summary><pre>' + _html.escape(txt) + '</pre></details>')

def _src_label(bundle):
    if bundle.images and bundle.images[0].source in ("Open Food Facts", "Open Products Facts"):
        chip = f'<span class="pg-badge pg-hit">{bundle.images[0].source} hit</span>'
    elif bundle.images and bundle.images[0].source == "sidecar":
        chip = '<span class="pg-badge pg-demo">demo image (sidecar)</span>'
    else:
        chip = '<span class="pg-badge pg-miss">no catalogue hit</span>'
    if getattr(bundle, "ambiguous", False):
        chip += ' <span class="pg-badge pg-miss">ambiguous match</span>'
    return chip

def _card(path, identity, bundle):
    i = identity
    cat_img = bundle.images[0].url if bundle.images else ""
    thumb = f'<img class="pg-thumb" src="{_img_src(cat_img)}"/>' if cat_img else ""
    ocr = (i.ocr_text or "")[:80]
    tags = ", ".join(bundle.category_tags[:4]) if bundle.category_tags else "-"
    price = f"{bundle.price.point} {bundle.price.currency}" if bundle.price.point else "-"
    return _CSS + f"""<div class="pg-card">
      <div class="pg-photo"><img src="{_img_src(path)}"/><div class="cap">{_html.escape(os.path.basename(path))}</div></div>
      <div class="pg-col"><div class="pg-h">Recognition (LLM)</div>
        <div class="pg-row"><b>brand</b> {_html.escape(i.brand or '-')}<br>
        <b>product</b> {_html.escape(i.product or '-')}<br>
        <b>variant</b> {_html.escape(i.variant or '-')}<br>
        <b>size</b> {_html.escape(i.size or '-')}<br>
        <b>category</b> {_html.escape(i.category or '-')}<br>
        <b>barcode</b> {_html.escape(str(i.barcode) or '-')}<br>
        <b>confidence</b> {i.confidence}<br>
        <b>search</b> {_html.escape(i.search_query or '-')}<br>
        <b>ocr</b> {_html.escape(ocr)}</div></div>
      <div class="pg-col">{_src_label(bundle)}<div class="pg-h">Retrieval (OFF / OPF)</div>
        <div class="pg-row"><b>canonical</b> {_html.escape(bundle.canonical_name or '-')}<br>
        <b>price</b> {_html.escape(price)}<br>
        <b>basis</b> {_html.escape(bundle.price.basis)}<br>
        <b>tags</b> {_html.escape(tags)}</div>{thumb}</div>
    </div>""" + _log_html(path)

def probe(path, show=True):
    """Recognition + retrieval only (no generate/guardrails). Writes <image>-log.txt too."""
    do_log = config.LOG_ENABLED
    if do_log:
        trace.start(f"image = {path} (recognition + retrieval only)")
    try:
        clean = preprocess(path)
        trace.log("CAPTURE", f"normalised -> {clean}")
        identity = recognise(clean)
        trace.log("RECOGNISE", "final identity", identity)
        bundle = retrieve(identity)
        trace.log("RETRIEVE", "final bundle", bundle)
    finally:
        if do_log:
            lp = os.path.join(os.path.dirname(path),
                              os.path.splitext(os.path.basename(path))[0] + "-log.txt")
            trace.dump(lp)
            trace.stop()
    if show:
        display(HTML(_card(path, identity, bundle)))
    return identity, bundle

# ---- direct OFF / OPF probes (no LLM) ----
def off_search(query, n=3):
    """Open Food Facts search (Search-a-licious). Returns list of hits."""
    r = requests.get(config.OFF_SEARCH_SALICIOUS,
                     params={"q": query, "page_size": n,
                             "fields": "code,product_name,brands,quantity,categories_tags,image_front_url"},
                     timeout=config.HTTP_TIMEOUT, headers={"User-Agent": config.USER_AGENT})
    r.raise_for_status()
    return r.json().get("hits", [])

def barcode_lookup(code, db="off"):
    """Look up a product by barcode. db = 'off' (food) or 'opf' (non-food)."""
    url = config.OFF_PRODUCT_URL if db == "off" else config.OPF_PRODUCT_URL
    r = requests.get(url.format(code=code), timeout=config.HTTP_TIMEOUT,
                     headers={"User-Agent": config.USER_AGENT})
    r.raise_for_status()
    d = r.json()
    return d.get("product") if d.get("status") == 1 else None


### Step 5 — recognition + retrieval on sample images

Each card: what recognition returned (left) vs what OFF/OPF have (right), with a hit/miss badge.

In [ ]:
for s in ["data/testset/duracell_aa.jpg", "data/testset/marlboro.jpg", "data/testset/flyinghorse.jpg"]:
    probe(s)

### Step 6 — upload your own images

In [ ]:
# Works in Colab (google.colab.files) and locally (ipywidgets fallback) — no zip needed.
os.makedirs("data/uploads", exist_ok=True)

def _save_uploads(items):
    paths = []
    for name, content in items:
        p = os.path.join("data/uploads", name)
        with open(p, "wb") as w:
            w.write(bytes(content))
        paths.append(p)
    return paths

try:
    from google.colab import files
    uploaded = files.upload()          # choose one or more images
    for p in _save_uploads(list(uploaded.items())):
        probe(p)
except ImportError:
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
        up = widgets.FileUpload(accept="image/*", multiple=True)
        out = widgets.Output()
        def _go(change):
            with out:
                clear_output()
                vals = up.value
                fs = list(vals.values()) if isinstance(vals, dict) else list(vals)
                items = [((f.get("name") or f.get("metadata", {}).get("name", "upload.jpg")), f["content"]) for f in fs]
                for p in _save_uploads(items):
                    probe(p)
        up.observe(_go, names="value")
        print("Upload product photos:")
        display(up, out)
    except ImportError:
        print("ipywidgets not installed -> pip install ipywidgets")

### Step 7 — direct OFF / OPF availability probe (no LLM)

Test the catalogue independently of the model — by **name** (Search-a-licious, food only) or by
**barcode** (food and non-food). Edit and re-run.

In [ ]:
# by NAME (food only):
try:
    for h in off_search("coca cola", n=3):
        print(h.get("code"), "|", h.get("product_name"), "|", h.get("brands"), "|", h.get("quantity"))
except Exception as e:
    print("OFF search unavailable:", e)

In [ ]:
# by BARCODE (db = "off" for food, "opf" for non-food e.g. batteries):
for code_str, db in [("5449000000996", "off"), ("5000394000000", "opf")]:
    try:
        p = barcode_lookup(code_str, db=db)
        print(db, code_str, "->", (p.get("product_name") if p else "not found"))
    except Exception as e:
        print(db, code_str, "-> error:", e)

### Notes
- Retrieval by **name** is food-only (OFF Search-a-licious); non-food (batteries, etc.) resolves
  only by **barcode**. "no catalogue hit" is expected for non-food name searches.
- `SNAP_STRICT_PROVIDER = True` disables the offline sidecar so the Recognition panel shows the
  real model output (a blank/"unknown" identity then means the API call failed).